# Análise exploratória e aplicação de modelos de Aprendizado de Máquina em acidentes rodoviários da PRF – 2026

**Integrantes:** Karlos Henryque e Paula Krupinski
**Fonte dos dados:** Polícia Rodoviária Federal (PRF) – Dados Abertos, *Acidentes 2026, agrupados por pessoa, todas as causas e tipos de acidente*.
https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf

> Todo o processamento é local. Nenhum dado é enviado a serviços externos.

## 1. Introdução

Acidentes em rodovias federais têm alto custo humano e econômico. A PRF disponibiliza os registros das ocorrências atendidas em formato aberto, o que permite caracterizar quando, onde e em que condições os acidentes acontecem e quais características se associam à gravidade das lesões das pessoas envolvidas.

## 2. Objetivos

1. Realizar a análise exploratória dos dados, identificando tendências, relações entre variáveis, variabilidade e valores extremos.
2. Preparar os dados para classificação, evitando vazamento de informação.
3. Treinar e comparar três modelos: **Árvore de Decisão**, **Regressão Logística** e **rede neural MLP**, para prever o **estado físico** de cada pessoa envolvida.
4. Discutir resultados e limitações com base apenas nos valores calculados.

## 3. Descrição da base de dados

In [ ]:
import sys, warnings
sys.path.append("../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import config as cfg, dados, estatistica as estat, visualizacao as viz, modelagem as ml, avaliacao as av

viz.aplicar_estilo()
pd.set_option("display.max_columns", 50, "display.width", 200, "display.float_format", "{:,.2f}".format)
cfg.RESULTADOS.mkdir(parents=True, exist_ok=True)

df = dados.carregar_bruto()
print(f"Linhas: {len(df):,} | Colunas: {df.shape[1]}")
print(f"Período: {df.data_inversa.min():%d/%m/%Y} a {df.data_inversa.max():%d/%m/%Y}")

: 

In [ ]:
diag = pd.DataFrame({"tipo": df.dtypes.astype(str), "nulos": df.isna().sum(),
                     "% nulos": 100 * df.isna().mean(), "únicos": df.nunique()})
diag.to_csv(cfg.RESULTADOS / "diagnostico_colunas.csv")
print("Linhas duplicadas (inteiras):", df.duplicated().sum())
diag

In [ ]:
df.select_dtypes('number').describe().T

### 3.1 Granularidade: linha, pessoa ou acidente?

O CSV é "agrupado por pessoa", mas cada pessoa pode aparecer em várias linhas (uma para cada combinação de causa e tipo de acidente). É preciso separar três unidades de contagem.

In [ ]:
com_pessoa = df[df.pesid.notna()]
por_pessoa = com_pessoa.groupby(["id", "pesid"]).size()
gran = pd.Series({
    "Linhas do CSV": len(df),
    "Linhas sem pessoa (pesid vazio)": df.pesid.isna().sum(),
    "Acidentes distintos (id)": df.id.nunique(),
    "Pessoas distintas (pesid)": com_pessoa.pesid.nunique(),
    "Pares (id, pesid) distintos": len(por_pessoa),
    "pesid em mais de um acidente": int(com_pessoa.drop_duplicates(["id","pesid"]).pesid.duplicated().sum()),
    "Acidentes sem nenhuma pessoa": int((df.groupby("id").pesid.apply(lambda s: s.notna().sum()) == 0).sum()),
})
print(gran.to_string(float_format="{:,.0f}".format))
print("\nLinhas por pessoa:"); print(por_pessoa.describe().round(2).to_string())
chk = com_pessoa.groupby(["id","pesid"]).agg(causas=("causa_acidente","nunique"), tipos=("tipo_acidente","nunique"), veic=("id_veiculo","nunique"))
print("\nMáximo por pessoa -> causas distintas:", chk.causas.max(), "| tipos distintos:", chk.tipos.max(), "| veículos:", chk.veic.max())
print("Pessoas por acidente:"); print(com_pessoa.drop_duplicates(["id","pesid"]).groupby("id").size().describe().round(2).to_string())

**Achados.** O CSV tem 457.622 linhas, mas apenas **135.786 pessoas** (`pesid`) e **54.895 acidentes** (`id`). Cada pessoa aparece em média 3,0 vezes (máximo 60), porque a linha se repete para cada combinação de causa × tipo de acidente (até 17 causas e 7 tipos distintos para a mesma pessoa). Nenhum `pesid` aparece em dois acidentes e cada pessoa está em um único veículo. Há 44.760 linhas sem pessoa (`pesid` vazio), mas todo acidente tem ao menos uma pessoa em outras linhas. Por acidente há em média 2,5 pessoas (mediana 2, máximo 87).

**Decisão.** Contar linhas inflaria os totais em cerca de 3,4 vezes em relação às pessoas e 8,3 vezes em relação aos acidentes. As análises de acidente usam uma linha por `id`; as de pessoa e a modelagem usam uma linha por (`id`, `pesid`). As colunas de data, UF, BR, fase do dia, clima, tipo de pista e horário são constantes dentro de cada `id`, o que valida a agregação.

In [ ]:
pessoas = dados.construir_pessoas(df)
# Unidade acidente: uma linha por id (colunas do acidente são constantes entre as linhas do mesmo id)
const = df.groupby("id")[["data_inversa","uf","br","fase_dia","condicao_metereologica","tipo_pista","dia_semana","horario"]].nunique().max()
print("Máx. de valores distintos por id (1 = constante):", const.to_dict())
acid = dados.construir_acidentes(df, pessoas)
print(f"Tabelas: pessoas={len(pessoas):,} | acidentes={len(acid):,}")

## 4. Metodologia

- **Unidades:** *acidente* (`id`), *pessoa* (`id`, `pesid`) e *linha* (pessoa × causa × tipo). Cada análise indica a unidade usada. Variáveis do acidente são analisadas na tabela de acidentes; variáveis da pessoa, na tabela de pessoas.
- **EDA:** Pandas, NumPy, Matplotlib e Seaborn. Associações entre categóricas: qui-quadrado e V de Cramér (o p-valor é pouco informativo com amostras grandes, então se prioriza o tamanho do efeito). Associação não implica causalidade.
- **Outliers:** regra do IQR (Tukey, k = 1,5), boxplots e inspeção do valor, sem remoção automática.
- **Modelagem:** Scikit-learn com `Pipeline` e `ColumnTransformer`; divisão treino/teste agrupada por acidente e estratificada; hiperparâmetros ajustados por validação cruzada agrupada apenas no treino, com F1 macro como critério.
- **Semente aleatória:** 42.

## 5. Análise exploratória

### 5.1 Tendências temporais e espaciais (unidade: acidente)

In [ ]:
dias = ["segunda-feira","terça-feira","quarta-feira","quinta-feira","sexta-feira","sábado","domingo"]
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
acid.groupby("mes").size().plot(marker="o", ax=ax[0])
m_ = acid.groupby("mes").size(); ax[0].set(title=f"Mês com mais acidentes: {['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set'][m_.idxmax()-1]} ({m_.max():,})", xlabel="Mês (2026)", ylabel="Nº de acidentes"); ax[0].set_ylim(0); ax[0].set_xticks(range(1, 10)); ax[0].set_xticklabels(["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set"])
acid.dia_semana.value_counts().reindex(dias).plot.bar(ax=ax[1], color=viz.AZUL)
d_ = acid.dia_semana.value_counts(); ax[1].set(title=f"Dia mais crítico: {d_.idxmax()}", xlabel="", ylabel="Nº de acidentes"); ax[1].tick_params(axis="x", rotation=40); viz.rotular(ax[1])
acid.groupby("hora").size().plot.bar(ax=ax[2], color=viz.AZUL)
h_ = acid.groupby("hora").size(); ax[2].set(title=f"Pico às {h_.idxmax()}h ({h_.max():,} acidentes)", xlabel="Hora do dia", ylabel="Nº de acidentes")
viz.salvar("01_tendencias_temporais")
print("Mensal:", acid.groupby("mes").size().to_dict())
print("Dia da semana:", acid.dia_semana.value_counts().reindex(dias).to_dict())
print("Horas de pico:", acid.groupby("hora").size().nlargest(3).to_dict())

**Interpretação.** Os dados cobrem **janeiro a setembro de 2026**, então não é possível falar de sazonalidade anual completa. O volume mensal é relativamente estável (de 5.598 em fevereiro a 6.406 em agosto, uma variação de cerca de 14%). Por dia da semana, **sábado (8.707) e domingo (8.740) têm cerca de 25% mais acidentes que a terça-feira (6.967)**, e a sexta-feira (8.442) também fica acima da média. Por hora, o maior volume ocorre às **18h (4.134 acidentes)**, seguido de 17h e 19h, com um pico secundário no início da manhã. Isso coincide com os horários de maior movimento nas rodovias; como não há dados de fluxo de veículos, não é possível separar maior exposição de maior risco.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
acid.uf.value_counts().head(10).plot.barh(ax=ax[0], color=viz.AZUL).invert_yaxis()
u_ = acid.uf.value_counts(); ax[0].set(title=f"{u_.index[0]} lidera: {100*u_.iloc[0]/len(acid):.0f}% dos acidentes", xlabel="Nº de acidentes", ylabel="UF"); viz.rotular(ax[0]); ax[0].set_xlim(0, u_.iloc[0]*1.15)
acid.groupby("br").size().nlargest(10).rename(lambda x: f"BR-{int(x)}").plot.barh(ax=ax[1], color=viz.AZUL).invert_yaxis()
ax[1].set(title="10 rodovias (BR) com mais acidentes", xlabel="Nº de acidentes", ylabel="Rodovia"); viz.rotular(ax[1]); ax[1].set_xlim(0, acid.br.value_counts().iloc[0]*1.15)
viz.salvar("02_uf_rodovia")
print((100 * acid.uf.value_counts(normalize=True).head(5)).round(1).to_dict())
print((100 * acid.br.value_counts(normalize=True).head(5)).round(1).to_dict(), "| BR ausente:", acid.br.isna().sum())

**Interpretação.** Cinco UFs (MG 13,0%, SC 11,5%, PR 10,7%, RJ 9,0% e RS 6,3%) concentram 50,5% dos acidentes. Duas rodovias respondem por 32% do total: **BR-101 (17,3%) e BR-116 (14,7%)**, bem acima da terceira, a BR-381 (4,9%). Volumes absolutos refletem a extensão das malhas e o tráfego de cada UF; por isso, comparações de gravidade entre estados usam taxas (seção 5.3), não contagens.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
acid.tipo_acidente.value_counts().head(10).plot.barh(ax=ax[0], color=viz.AZUL).invert_yaxis()
t_ = acid.tipo_acidente.value_counts(); ax[0].set(title=f"Tipo mais comum: {t_.index[0]} ({100*t_.iloc[0]/len(acid):.0f}%)", xlabel="Nº de acidentes", ylabel="Tipo de acidente (principal)"); viz.rotular(ax[0]); ax[0].set_xlim(0, t_.iloc[0]*1.15)
acid.condicao_metereologica.value_counts().plot.barh(ax=ax[1], color=viz.AZUL).invert_yaxis()
c_ = acid.condicao_metereologica.value_counts(); ax[1].set(title=f"{c_.index[0]} em {100*c_.iloc[0]/len(acid):.0f}% dos acidentes", xlabel="Nº de acidentes", ylabel="Condição meteorológica"); viz.rotular(ax[1]); ax[1].set_xlim(0, c_.iloc[0]*1.15)
viz.salvar("03_tipo_clima")
print((100*acid.tipo_acidente.value_counts(normalize=True).head(5)).round(1).to_dict())
print((100*acid.condicao_metereologica.value_counts(normalize=True).head(4)).round(1).to_dict())
print("Tipo de pista:", (100*acid.tipo_pista.value_counts(normalize=True)).round(1).to_dict())
print("Fase do dia:", (100*acid.fase_dia.value_counts(normalize=True)).round(1).to_dict())

**Interpretação.** A **colisão traseira** é o tipo principal mais frequente (20,2%), seguida de saída de leito carroçável (14,4%) e colisão transversal (12,8%). Em 62,6% dos acidentes o tempo estava com céu claro e em 10,0% chovia. Como o céu claro predomina no período, a frequência absoluta por clima não indica que esse clima seja mais perigoso; a comparação relevante é a proporção de tipos dentro de cada condição (seção 5.3). Quanto à via, 48,1% ocorreram em pista simples, 42,6% em dupla e 9,3% em múltipla; 54,9% em pleno dia e 34,7% em plena noite.

### 5.2 Perfil das pessoas envolvidas (unidade: pessoa)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
ev_ = pessoas.estado_fisico.value_counts(); ev_.plot.bar(ax=ax[0], color=[viz.COR[c] for c in ev_.index])
ax[0].set(title="Quase metade das pessoas saiu ilesa", xlabel="Estado físico", ylabel="Nº de pessoas"); ax[0].tick_params(axis="x", rotation=30); viz.rotular(ax[0])
sns.histplot(pessoas.idade.dropna().clip(upper=100), bins=50, ax=ax[1], color=viz.AZUL, edgecolor="white", linewidth=0.4)
ax[1].set(title="Distribuição da idade (valores > 100 agrupados em 100)", xlabel="Idade (anos)", ylabel="Nº de pessoas")
(pessoas.groupby(["sexo", "estado_fisico"]).size().unstack(0).pipe(lambda t: 100*t/t.sum())
   .loc[cfg.CLASSES + ["Não Informado"]].T).plot.bar(stacked=True, ax=ax[2], color=[viz.COR[c] for c in cfg.CLASSES + ["Não Informado"]])
ax[2].set(title="Estado físico por sexo (%)", xlabel="Sexo", ylabel="% das pessoas"); ax[2].tick_params(axis="x", rotation=0)
ax[2].legend(title="Estado físico", loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
viz.salvar("04_perfil_pessoas")
print((100*pessoas.estado_fisico.value_counts(normalize=True)).round(2).to_dict())
print((100*pessoas.sexo.value_counts(normalize=True)).round(1).to_dict())
print("Tipo de envolvido:", (100*pessoas.tipo_envolvido.value_counts(normalize=True)).round(1).to_dict())

**Interpretação.** Entre as 135.786 pessoas, 43,8% saíram ilesas, 35,3% com lesões leves, 11,0% com lesões graves, 3,4% morreram e 6,5% têm estado "Não Informado". Os envolvidos são 69,2% homens, 24,0% mulheres e 6,5% sem informação de sexo (coincidem com os "Não Informado" do estado físico); 68,5% eram condutores e 28,1% passageiros. A idade se concentra em adultos (mediana 38 anos), com leve assimetria à direita. O gráfico por sexo mostra que as mulheres têm mais lesões leves e os homens mais ilesos e uma proporção um pouco maior de óbitos; a categoria "Ignorado" (0,3% das pessoas) é pequena demais para conclusões.

### 5.3 Análise conjunta de variáveis

In [ ]:
conhecidos = pessoas[pessoas.estado_fisico.isin(cfg.CLASSES)].copy()
conhecidos["grave_obito"] = conhecidos.estado_fisico.isin(["Lesões Graves", "Óbito"])
conhecidos["faixa_idade"] = pd.cut(conhecidos.idade, [-1, 12, 17, 29, 44, 59, 110], labels=["0-12","13-17","18-29","30-44","45-59","60+"])

fig, ax = plt.subplots(1, 2, figsize=(17, 5))
h = pd.crosstab(conhecidos.hora, conhecidos.estado_fisico, normalize="index") * 100
for c in ["Lesões Graves", "Óbito"]:
    ax[0].plot(h.index, h[c], marker="o", lw=2, color=viz.COR[c], label=c)
ax[0].set(title=f"Madrugada concentra a maior % de óbitos (pico às {h['Óbito'].idxmax()}h: {h['Óbito'].max():.1f}%)", xlabel="Hora do dia", ylabel="% das pessoas na hora", ylim=(0, None))
ax[0].set_xticks(range(0, 24, 2)); ax[0].legend(title="Estado físico", frameon=False)
d = (conhecidos.groupby("dia_semana").grave_obito.mean().reindex(dias) * 100)
ax[1].bar(d.index, d.values, color=[viz.COR["Óbito"] if v > 100*conhecidos.grave_obito.mean() else "#B0B0B0" for v in d.values])
ax[1].axhline(100*conhecidos.grave_obito.mean(), ls="--", c="k", lw=1); viz.rotular(ax[1], "{:.1f}")
ax[1].set(title="Fim de semana tem maior % de lesão grave ou óbito (vinho = acima da média)", xlabel="Dia da semana", ylabel="% de pessoas com lesão grave ou óbito"); ax[1].tick_params(axis="x", rotation=40)
ax[1].text(6.4, 100*conhecidos.grave_obito.mean(), " média geral", va="bottom", ha="right", fontsize=9)
viz.salvar("05_hora_dia_gravidade")
print("Óbito+grave por dia (%):", d.round(2).to_dict()); print("Óbito por hora (%), 3 maiores:", h["Óbito"].nlargest(3).round(2).to_dict(), "| 3 menores:", h["Óbito"].nsmallest(3).round(2).to_dict())
print("Óbito+grave por fase do dia (%):", (100*conhecidos.groupby("fase_dia").grave_obito.mean()).round(2).to_dict())

**Interpretação.** A proporção de pessoas com lesão grave ou óbito é mais alta de madrugada: o **óbito chega a aproximadamente 8% às 3h e 4h**, contra 2% a 3% no período de 7h a 17h (valores lidos da tabulação cruzada hora × estado físico; ver também as saídas da célula). Por fase do dia, lesão grave ou óbito atinge 18,5% na plena noite e 18,3% no amanhecer, contra 13,3% em pleno dia. Por dia da semana, **sábado (16,4%) e domingo (16,9%) superam a média geral (cerca de 15,4%)**, enquanto a quarta-feira tem o menor valor (14,2%). A diferença entre dias é pequena (V de Cramér = 0,018 para pessoas) e, como não há controle de fluxo, velocidade ou tipo de veículo, não permite atribuir causa.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(17, 5.5))
(pd.crosstab(conhecidos.faixa_idade, conhecidos.estado_fisico, normalize="index").loc[:, cfg.CLASSES]*100).plot.bar(stacked=True, ax=ax[0], color=[viz.COR[c] for c in cfg.CLASSES])
ax[0].set(title="Estado físico por faixa etária (%)", xlabel="Faixa etária (anos)", ylabel="% das pessoas"); ax[0].tick_params(axis="x", rotation=0); ax[0].legend().remove()
topv = conhecidos.tipo_veiculo.value_counts().head(8).index
tv = pd.crosstab(conhecidos[conhecidos.tipo_veiculo.isin(topv)].tipo_veiculo, conhecidos.estado_fisico, normalize="index").loc[:, cfg.CLASSES]*100
tv.sort_values("Ileso").plot.barh(stacked=True, ax=ax[1], color=[viz.COR[c] for c in cfg.CLASSES])
ax[1].set(title="Motociclistas: poucos saem ilesos (8 veículos mais frequentes, %)", xlabel="% das pessoas", ylabel="Tipo de veículo"); ax[1].legend(title="Estado físico", loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
viz.salvar("06_idade_veiculo_estado")
print((100*conhecidos.groupby("faixa_idade", observed=True).grave_obito.mean()).round(2).to_dict())
print((100*conhecidos[conhecidos.tipo_veiculo.isin(topv)].groupby("tipo_veiculo").grave_obito.mean()).round(2).sort_values().to_dict())
print((100*conhecidos.groupby("tipo_envolvido").grave_obito.mean()).round(2).to_dict())

**Interpretação.** O tipo de veículo é a variável que mais separa os desfechos: **motocicleta (28,6%) e motoneta (25,2%)** têm as maiores proporções de lesão grave ou óbito, contra 6,9% a 10% para camioneta, ônibus, caminhão e automóvel. Entre os tipos de envolvido, **pedestres (64,9%) e cavaleiros (61,5%)** têm proporções muito superiores às de condutores (14,9%) e passageiros (13,5%), o que é coerente com a ausência de proteção física. Por faixa etária, a proporção varia de 11,2% (0–12 anos) a 17,3% (18–29 anos), com 16,4% em 60+; as diferenças existem mas são modestas.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(17, 6))
ct = pd.crosstab(acid.condicao_metereologica, acid.tipo_acidente, normalize="index") * 100
cols = acid.tipo_acidente.value_counts().head(8).index
sns.heatmap(ct.loc[acid.condicao_metereologica.value_counts().head(6).index, cols], annot=True, fmt=".0f", cmap="Blues", ax=ax[0], cbar_kws={"label": "% dentro da condição"})
ax[0].set(title="Condição climática × tipo de acidente (%)", xlabel="Tipo de acidente", ylabel="Condição meteorológica"); ax[0].tick_params(axis="x", rotation=40)
ct2 = pd.crosstab(acid.tipo_pista, acid.tipo_acidente, normalize="index") * 100
sns.heatmap(ct2[cols], annot=True, fmt=".0f", cmap="Blues", ax=ax[1], cbar_kws={"label": "% dentro do tipo de pista"})
ax[1].set(title="Tipo de pista × tipo de acidente (%)", xlabel="Tipo de acidente", ylabel="Tipo de pista"); ax[1].tick_params(axis="x", rotation=40); ax[1].tick_params(axis="y", rotation=0)
viz.salvar("07_clima_pista_tipo")

**Interpretação.** O tipo de acidente muda com as condições: sob **chuva, a saída de leito carroçável sobe para 27%** dos acidentes (12% com céu claro) e a colisão com objeto para 12% (6% no céu claro), enquanto a colisão traseira cai para 14% (21% no céu claro). Por tipo de pista, a **colisão frontal representa 12% dos acidentes em pista simples contra 1% a 2% em pistas dupla e múltipla**, e a colisão transversal chega a 19% na simples (7% nas demais). Nas pistas dupla e múltipla predominam colisão traseira (26% e 29%) e colisão lateral no mesmo sentido (14% e 21%). Esses padrões são consistentes com a existência de canteiro central e de mais de uma faixa por sentido.

In [ ]:
# Associações (V de Cramér = tamanho do efeito; 0 = nenhuma, 1 = perfeita)
pares = [("hora (pessoas)", conhecidos.hora, conhecidos.estado_fisico),
         ("dia_semana × estado_fisico", conhecidos.dia_semana, conhecidos.estado_fisico),
         ("faixa_idade × estado_fisico", conhecidos.faixa_idade, conhecidos.estado_fisico),
         ("tipo_veiculo × estado_fisico", conhecidos.tipo_veiculo.fillna("Ausente"), conhecidos.estado_fisico),
         ("tipo_envolvido × estado_fisico", conhecidos.tipo_envolvido, conhecidos.estado_fisico),
         ("sexo × estado_fisico", conhecidos.sexo, conhecidos.estado_fisico),
         ("uf × estado_fisico", conhecidos.uf, conhecidos.estado_fisico),
         ("tipo_pista × estado_fisico", conhecidos.tipo_pista, conhecidos.estado_fisico),
         ("condicao_meteo × tipo_acidente (acid.)", acid.condicao_metereologica, acid.tipo_acidente),
         ("tipo_pista × tipo_acidente (acid.)", acid.tipo_pista, acid.tipo_acidente),
         ("dia_semana × classificacao (acid.)", acid.dia_semana[acid.classificacao_acidente.notna()], acid.classificacao_acidente.dropna())]
res = []
for nome, a, b in pares:
    chi2, p, v = estat.cramers_v(a.values, b.values); res.append({"associação": nome.replace("hora (pessoas)", "hora × estado_fisico"), "qui²": chi2, "p-valor": p, "V de Cramér": v})
assoc = pd.DataFrame(res).set_index("associação").sort_values("V de Cramér", ascending=False)
assoc.to_csv(cfg.RESULTADOS / "associacoes_cramer.csv"); assoc.style.format({"qui²": "{:,.0f}", "p-valor": "{:.2e}", "V de Cramér": "{:.3f}"})

In [ ]:
# Idade (numérica) × estado físico: Kruskal-Wallis (não assume normalidade)
grupos = [g.idade.dropna() for _, g in conhecidos.groupby("estado_fisico")]
H, p = stats.kruskal(*grupos)
print(f"Kruskal-Wallis idade × estado_fisico: H = {H:,.1f}, p = {p:.2e}")
print(conhecidos.groupby("estado_fisico").idade.agg(["count","median","mean"]).loc[cfg.CLASSES].round(1))
# Estado × quantidade de ocorrências
uf = acid.groupby("uf").size().rename("acidentes").to_frame()
uf["pessoas_obito"] = conhecidos[conhecidos.estado_fisico=="Óbito"].groupby("uf").size()
uf["pessoas"] = conhecidos.groupby("uf").size(); uf["% óbito"] = 100*uf.pessoas_obito/uf.pessoas
print("\nUFs com maior taxa de óbito (UFs com >= 1000 pessoas):")
print(uf[uf.pessoas >= 1000].sort_values("% óbito", ascending=False).head(5).round(2))
print("\nUFs com menor taxa de óbito:"); print(uf[uf.pessoas >= 1000].sort_values("% óbito").head(5).round(2))

**Interpretação.** Com 126 mil pessoas, todos os testes qui-quadrado têm p < 0,001, então o p-valor discrimina pouco; o **V de Cramér** mede o tamanho do efeito. As associações mais fortes são **tipo de veículo × estado físico (V = 0,31)** e **tipo de pista × tipo de acidente (V = 0,28)**, seguidas de tipo de envolvido × estado físico (0,14). Idade (0,09), sexo (0,09), UF (0,07), tipo de pista (0,07), hora (0,06) e dia da semana (0,02) têm efeito pequeno sobre o estado físico. O Kruskal-Wallis também aponta diferença de idade entre os estados (H = 2.231; p < 0,001), mas as medianas variam pouco (35 a 42 anos); lesões leves ocorrem em pessoas mais jovens (mediana 35) e óbito em pessoas um pouco mais velhas (mediana 42, semelhante à dos ilesos, 41). A taxa de óbito por UF varia de **2,2% (DF, SP e SC) a 8,3% (MA)**, com PA (7,7%), CE (6,5%) e TO (5,7%) também altos; as causas dessa diferença (tipo de via, tempo de socorro, perfil de veículos, qualidade do registro) não podem ser identificadas com estes dados. Os resultados indicam associação, não causalidade.

### 5.4 Variabilidade

In [ ]:
tab_var = pd.concat([estat.variabilidade(pessoas, ["idade", "idade_veiculo"]),
                     estat.variabilidade(acid, ["km", "hora", "n_pessoas"])])
tab_var.index = ["idade (válida, 0-110)", "idade do veículo (anos)", "km da rodovia", "hora do acidente", "pessoas por acidente"]
tab_var.to_csv(cfg.RESULTADOS / "variabilidade.csv"); tab_var

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 4.5))
for a, (c, t, src) in zip(ax, [("idade","Idade (anos)",pessoas),("idade_veiculo","Idade do veículo (anos)",pessoas),("km","Km da rodovia",acid),("n_pessoas","Pessoas por acidente",acid)]):
    sns.boxplot(y=src[c], ax=a, color="lightsteelblue", flierprops={"markersize": 2}); a.set(title=f"{t}: mediana {src[c].median():.0f}", ylabel=t)
viz.salvar("08_boxplots_variabilidade")

**Interpretação.** Pelo coeficiente de variação (CV), a maior dispersão relativa é a de **pessoas por acidente (89,8%; assimetria 11,3)**, causada por poucos acidentes com muitas pessoas. Em seguida vêm **km da rodovia (87,3%)**, que reflete rodovias de extensões muito diferentes, e **idade do veículo (76,1%)**, com frota que vai de veículos novos a muito antigos (mediana 10 anos, Q3 = 16). A hora do acidente (CV 47,0%) e a idade da pessoa (CV 39,6%; desvio 15,7 anos; mediana 38) são mais homogêneas. Variância e amplitude dependem da unidade e só são comparáveis entre variáveis pelo CV.

### 5.5 Outliers

In [ ]:
raw = {"idade (bruta, pessoas)": df.drop_duplicates(["id","pesid"]).idade,
       "idade do veículo (corrigida)": pessoas.idade_veiculo, "km (acidentes)": acid.km, "pessoas por acidente": acid.n_pessoas}
out = pd.DataFrame({k: estat.outliers_iqr(v) for k, v in raw.items()}).T
out.to_csv(cfg.RESULTADOS / "outliers_iqr.csv"); print(out.round(2).to_string())
idade_bruta = raw["idade (bruta, pessoas)"]
print("\nIdades > 110:", int((idade_bruta > 110).sum()), "| valores:", sorted(idade_bruta[idade_bruta > 110].unique())[:10])
print("Idades 90-110:", int(idade_bruta.between(90, 110).sum()), "| Idade = 0:", int((idade_bruta == 0).sum()))
fab = df.drop_duplicates(["id", "id_veiculo"]).ano_fabricacao_veiculo
print("Ano de fabricação: mín", fab.min(), "máx", fab.max(), "| > 2027:", int((fab > 2027).sum()), "| < 1950:", int((fab < 1950).sum()))
print("km máx:", acid.km.max(), "| acidentes com 30+ pessoas:", int((acid.n_pessoas >= 30).sum()), "| máx pessoas:", acid.n_pessoas.max())

**Critério:** regra de Tukey (limites Q1 − 1,5·IQR e Q3 + 1,5·IQR). Nenhum registro foi removido; cada caso foi avaliado:

| Variável | Outliers (IQR) | Decisão |
|---|---|---|
| **Idade (bruta)** | 416 (0,34%); limite superior = 83 anos | Idades de 83 a 110 são plausíveis e foram mantidas (70 pessoas com 90–110 anos). **42 valores > 110** (126, 915, 927, 1.010, 1.915, 2.025 etc.) são **erros de digitação** (alguns parecem anos) e viraram ausentes, imputados pela mediana no pipeline. Idade 0 (89 casos) é plausível (bebês). |
| **Ano de fabricação** | 7.820 veículos com ano < 1950 | O valor **0** é código de dado ausente (mínimo = 0); idade de veículo > 80 anos virou ausente. |
| **Idade do veículo** | 1.597 (1,3%); limite = 34 anos | Veículos antigos são legítimos; mantidos. |
| **Km da rodovia** | 581 (1,06%); limite = 903,8 | Máximo 1.260 km, compatível com rodovias longas; mantidos. |
| **Pessoas por acidente** | 1.214 (2,2%); limite = 6 | 73 acidentes com 30 ou mais pessoas (máximo 87) são extremos mas possíveis (ex.: ônibus); não verificados caso a caso, mantidos por serem observações legítimas da população. |

Os outliers legítimos permanecem porque representam acidentes graves reais, que justamente interessam à análise.

## 6. Preparação dos dados

### 6.1 Definição do problema
**Classificação multiclasse** do `estado_fisico` de cada pessoa (Ileso, Lesões Leves, Lesões Graves, Óbito), com base em características do acidente, do veículo e da pessoa.

**Decisões metodológicas**
1. **Unidade:** pessoa (`id`, `pesid`), 135.786 registros, para não contar a mesma pessoa várias vezes por combinar causa × tipo.
2. **"Não Informado"** (rótulo desconhecido) é excluído da modelagem. Ele não é uma classe clínica, e prever "não informado" indicaria apenas falha de registro.
3. **Vazamento (data leakage):** `ilesos`, `feridos_leves`, `feridos_graves` e `mortos` são exatamente o `estado_fisico` recodificado (verificado por tabulação cruzada, ver abaixo). `classificacao_acidente` resume o pior desfecho entre os envolvidos, o que inclui o alvo. As cinco colunas são excluídas. Colunas de identificação (`id`, `pesid`, `id_veiculo`) e de localização fina (`municipio`, `latitude`, `longitude`, `delegacia`, `uop`, `regional`, `marca`) não entram: são de alta cardinalidade ou redundantes com `uf`/`km`/`br`.
4. **Idades > 110 e idade de veículo fora de [0, 80]** foram convertidas em ausentes (erros de registro), e imputadas pela mediana dentro do pipeline, com indicador de ausência.
5. **Divisão:** `StratifiedGroupKFold` (20% para teste), agrupada por acidente, para que pessoas do mesmo acidente não fiquem nos dois conjuntos, e estratificada pela classe.

In [ ]:
print(pd.crosstab(pessoas.estado_fisico, [pessoas.ilesos, pessoas.feridos_leves, pessoas.feridos_graves, pessoas.mortos],
                  rownames=["estado_fisico"], colnames=["ilesos","leves","graves","mortos"]))

In [ ]:
X, y, grupos = ml.dados_modelagem(pessoas)
tr, te = ml.dividir(X, y, grupos)
Xtr, Xte, ytr, yte, gtr = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te], grupos.iloc[tr]
print(f"Registros modelados: {len(X):,} | Treino: {len(Xtr):,} | Teste: {len(Xte):,}")
print("Acidentes em comum entre treino e teste:", len(set(grupos.iloc[tr]) & set(grupos.iloc[te])))
dist = pd.DataFrame({"treino %": 100*ytr.value_counts(normalize=True), "teste %": 100*yte.value_counts(normalize=True), "treino n": ytr.value_counts()}).loc[cfg.CLASSES]
dist.to_csv(cfg.RESULTADOS / "distribuicao_classes.csv"); print(dist.round(2).to_string())
print("\nPreditoras numéricas:", ml.NUMERICAS); print("Preditoras categóricas:", ml.CATEGORICAS)
print("\nAusentes nas preditoras:", X.isna().sum()[lambda s: s > 0].to_dict())
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=dist.index, y=dist["treino n"], color=viz.AZUL, ax=ax); ax.set(title="Distribuição das classes no treino", xlabel="Estado físico", ylabel="Nº de pessoas")
viz.salvar("09_classes_treino")

**Resultado.** Após excluir "Não Informado" (8.793 pessoas), restam **126.993 pessoas** (treino 101.594; teste 25.399), sem nenhum acidente em comum entre os dois conjuntos. As proporções de classe são praticamente idênticas nos dois: Ileso 46,8%, Lesões Leves 37,8%, Lesões Graves 11,8% e **Óbito 3,6%** (razão de 13 para 1 entre a maior e a menor classe). O desbalanceamento justifica a ponderação por classe e a escolha de **F1 macro** (e não a acurácia) como critério. A tabulação cruzada confirma o vazamento: cada combinação de `ilesos`, `feridos_leves`, `feridos_graves` e `mortos` corresponde a exatamente um estado físico.

**Limitação de desenho:** as preditoras descrevem o acidente já ocorrido (tipo de acidente, veículo, local), portanto o modelo é **descritivo**: associa características do acidente à gravidade, não prevê o risco antes de ele acontecer.

## 7. Modelos de Aprendizado de Máquina

Os três modelos usam o mesmo pré-processamento (`ColumnTransformer`): imputação pela mediana com indicador de ausência para as numéricas e *one-hot encoding* para as categóricas (categorias com menos de 50 ocorrências são agrupadas). A padronização (`StandardScaler`) é aplicada **ao MLP e à regressão logística**, que são sensíveis à escala; as árvores são invariantes a ela.

Para lidar com o desbalanceamento, os três usam ponderação inversa à frequência da classe (`class_weight="balanced"` na árvore e na regressão logística, e `sample_weight` no MLP). O ajuste de hiperparâmetros usa `RandomizedSearchCV` com validação cruzada agrupada (3 dobras por acidente) e F1 macro como critério, em uma subamostra de 40% dos acidentes do **treino** (para limitar o tempo). Depois, o melhor modelo é reajustado em todo o treino. **O teste só é usado na avaliação final.**

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

Xs, ys, cv_idx = ml.subamostra_busca(Xtr, ytr, gtr, frac=0.4)
busca, treino = (Xs, ys, cv_idx), (Xtr, ytr)
print(f"Subamostra de ajuste: {len(Xs):,} pessoas de {gtr.loc[Xs.index].nunique():,} acidentes")


### 7.1 Modelo 1 – Árvore de Decisão (`DecisionTreeClassifier`)
Divide recursivamente os dados com regras do tipo "variável ≤ limiar" (ou "categoria = sim"), escolhendo a divisão que mais reduz a impureza (Gini ou entropia). É **interpretável**, mas uma árvore profunda memoriza o treino (alta variância). Por isso a profundidade (`max_depth`) e o tamanho mínimo da folha (`min_samples_leaf`) são ajustados.

In [ ]:
arvore, busca_arv = ml.ajustar("arvore_decisao", DecisionTreeClassifier(class_weight="balanced", random_state=cfg.SEED),
    {"max_depth": [4, 6, 8, 10, 14, 20], "criterion": ["gini", "entropy"], "min_samples_leaf": [10, 30, 100]}, n_iter=14, escalar=False, busca=busca, treino=treino)
print("Profundidade real:", arvore[-1].get_depth(), "| folhas:", arvore[-1].get_n_leaves())
busca_arv[["parametros", "mean_test_score", "std_test_score"]].head(5)

In [ ]:
nomes = arvore[0].get_feature_names_out()
fig, ax = plt.subplots(figsize=(24, 9))
plot_tree(arvore[-1], max_depth=3, feature_names=[n.replace("cat__","").replace("num__","") for n in nomes], class_names=list(arvore[-1].classes_),
          filled=True, rounded=True, fontsize=7, proportion=True, impurity=False, ax=ax)
ax.set_title("Árvore de Decisão – três primeiros níveis (modelo final tem mais níveis)")
viz.salvar("10_arvore_decisao")

### 7.2 Modelo 2 – Regressão Logística (`LogisticRegression`)
Classificador **linear**: para cada classe calcula uma pontuação que é uma soma ponderada das preditoras (após a padronização e o one-hot), e a transforma em probabilidades com a função softmax (uma por classe). Os pesos são estimados por máxima verossimilhança, com regularização L2 controlada por `C` (valores menores = regularização mais forte). É simples e interpretável pelos coeficientes, mas só traça fronteiras lineares: não captura interações entre variáveis, a menos que elas sejam criadas manualmente.

In [ ]:
lr, busca_lr = ml.ajustar("regressao_logistica", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=cfg.SEED),
    {"C": [0.01, 0.1, 1, 10]}, n_iter=4, escalar=True, busca=busca, treino=treino)
busca_lr[["parametros", "mean_test_score", "std_test_score"]].head(5)

### 7.3 Modelo 3 – Rede neural MLP (`MLPClassifier`)
*Perceptron* multicamadas: a **camada de entrada** tem um neurônio por atributo pré-processado (numéricas padronizadas + colunas *one-hot*); as **camadas ocultas** aplicam combinações lineares seguidas de ativação não linear (**ReLU**); a **camada de saída** usa **softmax** (uma probabilidade por classe). Os pesos são otimizados por **Adam** (gradiente estocástico adaptativo) minimizando a entropia cruzada, com regularização L2 (`alpha`). O treino termina por **parada antecipada**: separa 10% do treino como validação interna e para quando a pontuação não melhora por 10 épocas consecutivas (ou ao atingir 200 épocas). Diferentemente das árvores e da regressão logística, o MLP aprende uma fronteira de decisão suave e contínua, é sensível à escala (daí a padronização) e é uma "caixa-preta".

In [ ]:
mlp, busca_mlp = ml.ajustar("mlp", MLPClassifier(activation="relu", solver="adam", early_stopping=True, validation_fraction=0.1,
                                              n_iter_no_change=10, max_iter=200, random_state=cfg.SEED),
    {"hidden_layer_sizes": [(64,), (128,), (128, 64), (64, 32)], "alpha": [1e-4, 1e-3, 1e-2], "learning_rate_init": [1e-3, 3e-3]},
    n_iter=8, escalar=True, peso=True, busca=busca, treino=treino)
m = mlp[-1]
print(f"Camada de entrada: {mlp[0].transform(Xte.iloc[:5]).shape[1]} neurônios | ocultas: {m.hidden_layer_sizes} | saída: {m.n_outputs_} (softmax) | épocas efetivas: {m.n_iter_}")
busca_mlp[["parametros", "mean_test_score", "std_test_score"]].head(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(m.loss_curve_); ax[0].set(title="MLP – perda no treino", xlabel="Época", ylabel="Entropia cruzada")
ax[1].plot(m.validation_scores_); ax[1].set(title="MLP – pontuação de validação interna (acurácia)", xlabel="Época", ylabel="Acurácia")
viz.salvar("11_mlp_curvas")

**Resultados do ajuste (validação cruzada agrupada no treino):**

| Modelo | Melhores hiperparâmetros | F1 macro (CV) |
|---|---|---|
| Árvore de Decisão | `max_depth = 6`, `min_samples_leaf = 30`, critério Gini (58 folhas) | 0,414 ± 0,011 |
| Regressão Logística | `C = 0,1` (regularização L2) | 0,445 ± 0,003 |
| MLP | 1 camada oculta de 64 neurônios, `alpha = 1e-4`, taxa de aprendizado 0,003 | 0,447 ± 0,004 |

- **Árvore:** profundidades de 4 a 14 obtiveram F1 macro quase idêntico na validação (≈0,41); a árvore rasa (profundidade 6) foi escolhida, o que a mantém interpretável. A árvore profunda não trouxe ganho.
- **Regressão Logística:** os quatro valores de `C` testados (0,01 a 10) tiveram F1 macro quase idêntico (0,440 a 0,445), e `C = 0,1` foi escolhido. Com as variáveis padronizadas, a força da regularização tem pouco efeito.
- **MLP:** entrada com **301 neurônios** (numéricas padronizadas + colunas *one-hot*), **1 camada oculta de 64 neurônios (ReLU)** e **4 neurônios de saída (softmax)**. O treino parou em **20 épocas** pela parada antecipada. Entre as 8 configurações sorteadas, a melhor foi a de uma camada oculta; não se afirma que redes maiores sejam piores em geral.
- A diferença entre as três dobras (desvio ≤ 0,011) é da mesma ordem da diferença entre a Regressão Logística e o MLP (0,002). Por isso, esses dois modelos empatam na validação cruzada.

## 8. Avaliação e comparação (conjunto de teste, mesmas condições para os três modelos)

In [ ]:
modelos = {"Árvore de Decisão": arvore, "Regressão Logística": lr, "MLP": mlp}
metricas, preds, rel = {}, {}, {}
for nome, mod in modelos.items():
    metricas[nome], preds[nome] = av.avaliar(mod, Xte, yte); rel[nome] = av.por_classe(yte, preds[nome])
tabela = pd.DataFrame(metricas).T
tabela.to_csv(cfg.RESULTADOS / "comparacao_modelos.csv")
pd.concat(rel, names=["modelo"]).to_csv(cfg.RESULTADOS / "metricas_por_classe.csv")
tabela.style.format("{:.3f}").highlight_max(color="lightgreen")

In [ ]:
for nome in modelos:
    print(f"\n== {nome} =="); print(rel[nome].round(3).to_string())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(19, 5.5))
for a, nome in zip(ax, modelos): av.matriz(yte, preds[nome], a, nome)
viz.salvar("12_matrizes_confusao")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(17, 5))
tabela[["Acurácia", "Balanced Accuracy", "F1 macro", "F1 ponderado"]].plot.bar(ax=ax[0], rot=0, color=["#4C72B0", "#DD8452", "#55A868", "#8172B3"]); [ax[0].bar_label(c, fmt="%.2f", fontsize=8, padding=2) for c in ax[0].containers]
ax[0].set(title="Métricas globais no teste", xlabel="Modelo", ylabel="Valor (0 a 1)", ylim=(0, 1)); ax[0].legend(loc="lower right")
pd.concat({n: r["recall"] for n, r in rel.items()}, axis=1).loc[cfg.CLASSES].plot.bar(ax=ax[1], rot=0, color=["#4C72B0", "#DD8452", "#55A868"]); [ax[1].bar_label(c, fmt="%.2f", fontsize=8, padding=2) for c in ax[1].containers]
ax[1].set(title="Recall por classe (teste)", xlabel="Classe", ylabel="Recall"); ax[1].legend(loc="upper right")
viz.salvar("13_comparacao_modelos")

In [ ]:
from sklearn.metrics import roc_curve
yb = yte.isin(["Lesões Graves", "Óbito"])
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for nome, mod in modelos.items():
    cls = list(mod.classes_); s = mod.predict_proba(Xte)[:, [cls.index("Lesões Graves"), cls.index("Óbito")]].sum(1)
    fpr, tpr, _ = roc_curve(yb, s); ax.plot(fpr, tpr, label=f"{nome} (AUC = {tabela.loc[nome, 'AUC grave/óbito']:.3f})")
ax.plot([0, 1], [0, 1], "k--", label="Aleatório"); ax.legend(loc="lower right")
ax.set(title="Curva ROC – lesão grave ou óbito vs. demais", xlabel="Taxa de falsos positivos", ylabel="Taxa de verdadeiros positivos")
viz.salvar("14_roc_grave_obito")

### 8.1 Importância das variáveis (permutação, F1 macro, subamostra do teste)

In [ ]:
from sklearn.inspection import permutation_importance
amostra = Xte.sample(6000, random_state=cfg.SEED); ya = yte.loc[amostra.index]
imp = {}
for nome, mod in modelos.items():
    r = permutation_importance(mod, amostra, ya, scoring="f1_macro", n_repeats=3, random_state=cfg.SEED, n_jobs=-1)
    imp[nome] = pd.Series(r.importances_mean, index=amostra.columns)
imp = pd.DataFrame(imp); imp.to_csv(cfg.RESULTADOS / "importancia_permutacao.csv")
top = imp.mean(axis=1).nlargest(10).index
imp.loc[top].iloc[::-1].plot.barh(figsize=(10, 6))
plt.title("Queda no F1 macro ao embaralhar a variável (10 mais importantes)"); plt.xlabel("Queda média no F1 macro"); plt.ylabel("Variável")
viz.salvar("15_importancia_variaveis")
print(imp.round(4).loc[top].to_string())
for nome in imp: print(nome, "->", list(imp[nome].nlargest(5).index))

In [ ]:
# Estabilidade: dispersão do F1 macro entre as dobras de validação cruzada do melhor modelo de cada tipo
est = pd.DataFrame({n: {"F1 macro CV (média)": b.mean_test_score.iloc[0], "desvio entre dobras": b.std_test_score.iloc[0]}
                    for n, b in {"Árvore de Decisão": busca_arv, "Regressão Logística": busca_lr, "MLP": busca_mlp}.items()}).T
est["F1 macro no teste"] = tabela["F1 macro"]; est.to_csv(cfg.RESULTADOS / "estabilidade.csv"); est.round(4)

**Resultados no teste (25.399 pessoas, mesmas condições para os três modelos):**

| Modelo | Acurácia | Balanced Acc. | F1 macro | F1 pond. | AUC OvR | AUC grave/óbito |
|---|---|---|---|---|---|---|
| Árvore de Decisão | 0,574 | 0,503 | 0,437 | 0,572 | 0,757 | 0,772 |
| **Regressão Logística** | **0,584** | **0,538** | **0,462** | **0,600** | **0,801** | **0,804** |
| MLP | 0,571 | 0,523 | 0,454 | 0,594 | 0,785 | 0,800 |

**Por classe (recall / F1):**

| Classe | Árvore | Regressão Logística | MLP |
|---|---|---|---|
| Ileso | 0,82 / 0,76 | 0,76 / 0,77 | 0,72 / 0,75 |
| Lesões Leves | 0,37 / 0,47 | 0,44 / 0,52 | 0,46 / 0,52 |
| Lesões Graves | 0,27 / 0,28 | 0,33 / 0,30 | 0,34 / 0,29 |
| Óbito | 0,56 / 0,25 | 0,62 / 0,26 | 0,57 / 0,25 |

**Leitura.** A **Regressão Logística** tem o melhor conjunto de métricas globais (acurácia 0,584, F1 macro 0,462, AUC 0,801 e 0,804), à frente do MLP e da Árvore. A vantagem sobre o MLP é pequena (F1 macro 0,462 contra 0,454), o que sugere que a não linearidade do MLP traz pouco ganho com estas variáveis. O desempenho é moderado: nenhuma classe minoritária passa de F1 = 0,30. A **precisão do Óbito é baixa (0,16 em todos os modelos)**: os modelos encontram de 56% a 62% dos óbitos, mas ao custo de muitos falsos positivos (um "óbito" previsto só é real em cerca de 16% dos casos). A Regressão Logística tem o maior recall de óbito (0,62) e a melhor F1 em lesões graves (0,30); o MLP tem o maior recall de lesões graves (0,34), e a F1 em lesões leves é praticamente igual entre a regressão e o MLP (0,52). A confusão se concentra entre classes vizinhas (Leve ↔ Grave ↔ Óbito), como esperado em uma escala ordinal. Para separar "lesão grave ou óbito" dos demais, as AUCs de 0,77 a 0,80 indicam capacidade de discriminação razoável, bem acima do acaso (0,50).

**Importância das variáveis (queda no F1 macro ao embaralhar).** Nos três modelos, **`tipo_veiculo` é a mais importante (queda de 0,13 a 0,14)** e `tipo_acidente_principal` é a segunda (≈0,05 a 0,06), com ordem idêntica entre os modelos. As demais contribuem pouco (≤ 0,03): idade do veículo na árvore (0,03), tipo de envolvido, causa principal e UF na regressão logística (≈0,02) e número de tipos de acidente no MLP (0,01). Isso é coerente com a análise exploratória, em que tipo de veículo apresentou o maior V de Cramér (0,31). **Estabilidade:** o desvio do F1 macro entre as dobras foi de 0,003 (regressão logística), 0,004 (MLP) e 0,011 (Árvore). O F1 macro do teste é um pouco maior que o da validação cruzada (0,46 vs. 0,45 na regressão logística) porque o modelo final foi treinado com todo o treino, e não com a subamostra de 40% usada na busca.

## 9. Discussão dos resultados

1. **Tendências.** Mais acidentes em sábados e domingos (≈25% acima da terça-feira), pico às 18h, volume mensal estável (jan–set), 32% dos acidentes em duas rodovias (BR-101 e BR-116), colisão traseira como tipo mais comum (20,2%) e tempo com céu claro em 62,6% dos casos.
2. **Associações relevantes.** Tipo de veículo × estado físico (V = 0,31; motocicleta 28,6% e motoneta 25,2% com lesão grave ou óbito, contra 6,9% a 10% nos demais), tipo de pista × tipo de acidente (V = 0,28; colisão frontal em 12% dos acidentes de pista simples) e tipo de envolvido (pedestre 64,9%). Idade, sexo, UF, hora e dia da semana têm efeito pequeno (V ≤ 0,09). A chuva muda o tipo de acidente (saída de pista 27% vs. 12%). Nenhum resultado indica causalidade.
3. **Maior variabilidade.** Pessoas por acidente (CV 89,8%), km da rodovia (87,3%) e idade do veículo (76,1%); a menor é a da idade da pessoa (39,6%).
4. **Outliers.** 42 idades impossíveis (> 110) e o ano de fabricação "0" em 7.820 veículos (erros de registro, tratados como ausentes); idades de 83 a 110 anos, veículos antigos, km até 1.260 e acidentes de até 87 pessoas foram considerados legítimos e mantidos.
5. **Melhor modelo.** Regressão Logística (acurácia 0,584; F1 macro 0,462; AUC 0,801), à frente do MLP (0,454) e da Árvore (0,437). A diferença para o MLP é pequena, e o modelo linear é o mais simples de interpretar.
6. **Classes minoritárias.** O maior recall de óbito é da Regressão Logística (0,62), e o maior de lesões graves é do MLP (0,34); a Regressão Logística tem a melhor F1 em lesões graves (0,30). Todos têm precisão para óbito de cerca de 0,16.
7. **Variáveis mais importantes.** Tipo de veículo e tipo de acidente principal, nos três modelos.
8. **Limitações.** (i) Os dados cobrem jan–set de 2026 e uma única fonte. (ii) As preditoras só descrevem o acidente depois de ocorrido; faltam variáveis fortemente ligadas à gravidade (velocidade, uso de cinto ou capacete, ponto de impacto), o que limita o desempenho (F1 macro ≤ 0,46, F1 ≤ 0,30 nas classes minoritárias). (iii) 6,5% das pessoas (Não Informado) foram excluídas da modelagem, e podem não ser aleatórias. (iv) O ajuste usou 40% dos acidentes do treino e uma busca aleatória pequena (4 a 14 combinações), então os hiperparâmetros podem não ser ótimos. (v) A importância por permutação usou 6.000 pessoas do teste e variáveis correlacionadas podem dividir importância. (vi) Dados de registro policial podem ter erros e subnotificação.
9. **Trabalhos futuros.** Agregar dados externos (volume de tráfego, tipo de via, velocidade máxima); testar gradient boosting; tratar o alvo como ordinal ou como problema binário (grave/óbito); ajustar o limiar de decisão para recall ou precisão desejados; busca de hiperparâmetros mais ampla com todo o treino; estudar a variabilidade por UF com modelos multinível.

## 10. Conclusão

A base da PRF para 2026 (jan–set) contém 54.895 acidentes e 135.786 pessoas; separar essas unidades foi essencial, pois as linhas do CSV repetem cada pessoa em média 3 vezes. A análise exploratória mostrou concentração de acidentes em fins de semana, no fim da tarde e em duas rodovias, e que o **tipo de veículo** (motocicletas e motonetas) e o **tipo de envolvido** (pedestres) são os fatores mais associados à gravidade, enquanto idade, sexo, hora e dia da semana têm efeito pequeno.

Na previsão do estado físico, a **Regressão Logística** teve o melhor desempenho (F1 macro 0,462; acurácia 0,584; AUC 0,801), seguido do MLP (0,454) e da Árvore de Decisão (0,437). O desempenho é moderado e as classes minoritárias (lesões graves e óbito) são difíceis de identificar (F1 ≤ 0,30), pois as informações disponíveis não incluem os principais determinantes físicos da gravidade. Os resultados descrevem associações presentes nos dados e não permitem afirmar relações causais.

## 11. Referências

- BREIMAN, L.; FRIEDMAN, J.; OLSHEN, R.; STONE, C. *Classification and Regression Trees*. Belmont: Wadsworth, 1984.
- CRAMÉR, H. *Mathematical Methods of Statistics*. Princeton: Princeton University Press, 1946.
- GOODFELLOW, I.; BENGIO, Y.; COURVILLE, A. *Deep Learning*. Cambridge: MIT Press, 2016.
- KINGMA, D. P.; BA, J. Adam: a method for stochastic optimization. In: *International Conference on Learning Representations (ICLR)*, 2015.
- KRUSKAL, W. H.; WALLIS, W. A. Use of ranks in one-criterion variance analysis. *Journal of the American Statistical Association*, v. 47, n. 260, p. 583-621, 1952.
- PEDREGOSA, F. et al. Scikit-learn: machine learning in Python. *Journal of Machine Learning Research*, v. 12, p. 2825-2830, 2011.
- POLÍCIA RODOVIÁRIA FEDERAL (PRF). *Dados abertos da PRF – Acidentes 2026 (agrupados por pessoa)*. Disponível em: https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf.
- TUKEY, J. W. *Exploratory Data Analysis*. Reading: Addison-Wesley, 1977.
- WASKOM, M. L. Seaborn: statistical data visualization. *Journal of Open Source Software*, v. 6, n. 60, 3021, 2021.